# Percentage-Change Distribution Analysis

This notebook is the percentage-change counterpart to `analysis.ipynb`. It loads the cleaned CHronoZ macro series and studies the distribution of ordinary percentage changes, using the same source observations rather than changing the original notebook.

For each series, the transformation is:

\\[\\text{pct\_change}_t = 100 \\times \\left(\\frac{X_t}{X_{t-1}} - 1\\right)\\]

This is **ordinary percentage change**, not log change. A percentage change is undefined when the previous observation is zero. Values whose previous observation is zero are therefore marked invalid rather than silently converted to zero. Negative or sign-crossing series need special economic interpretation; diagnostics are retained so these cases are visible. The first observation in every series is naturally unavailable after differencing.

This is an exploratory comparison of distribution shape, not a declaration that percentage change is suitable for every macro indicator.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)
plt.rcParams['figure.figsize'] = (12, 5)

# Resolve paths from the repository root or from this notebook's directory.
CANDIDATE_DATA_DIRS = [Path('data/cleaned'), Path('../../data/cleaned'), Path('../data/cleaned')]
DATA_DIR = next((p for p in CANDIDATE_DATA_DIRS if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Could not find data/cleaned. Run from the repository root or notebook/deltaAnalysis/.')

print(f'Using cleaned data directory: {DATA_DIR.resolve()}')

## 1. Load and standardize the cleaned series

The loader detects the date column and a numeric observation column from each cleaned CSV. It preserves the original series values and only creates a transformed copy for this analysis. If a file has an ambiguous schema, the loader raises an error instead of guessing silently.

In [ ]:
def load_series(csv_path: Path) -> pd.Series:
    frame = pd.read_csv(csv_path)
    if frame.empty:
        raise ValueError(f'{csv_path.name} is empty')

    normalized = {str(c).strip().lower(): c for c in frame.columns}
    date_candidates = ['observation_date', 'date', 'observation date', 'time']
    date_col = next((normalized[c] for c in date_candidates if c in normalized), None)
    if date_col is None:
        date_col = frame.columns[0]

    value_candidates = ['value', 'observation', 'close', 'series_value']
    value_col = next((normalized[c] for c in value_candidates if c in normalized and normalized[c] != date_col), None)
    if value_col is None:
        numeric_candidates = [c for c in frame.columns if c != date_col and pd.to_numeric(frame[c], errors='coerce').notna().sum() > 0]
        if len(numeric_candidates) != 1:
            raise ValueError(f'{csv_path.name}: expected one numeric value column; found {numeric_candidates}')
        value_col = numeric_candidates[0]

    dates = pd.to_datetime(frame[date_col], errors='coerce')
    values = pd.to_numeric(frame[value_col], errors='coerce')
    result = pd.Series(values.to_numpy(), index=dates, name=csv_path.stem)
    result = result[~result.index.isna()].sort_index()
    result = result[~result.index.duplicated(keep='last')]
    return result.astype(float)

csv_files = sorted(DATA_DIR.glob('*.csv'))
csv_files = [p for p in csv_files if p.stem != 'series_metadata']
series_data = {}
load_errors = {}
for path in csv_files:
    try:
        series_data[path.stem] = load_series(path)
    except Exception as exc:
        load_errors[path.stem] = str(exc)

print(f'Series loaded: {len(series_data)}')
print(f'Files skipped due to schema/load errors: {len(load_errors)}')
if load_errors:
    display(pd.DataFrame([{'series': k, 'error': v} for k, v in load_errors.items()]))
display(pd.DataFrame([{'series': k, 'observations': v.notna().sum(), 'start': v.first_valid_index(), 'end': v.last_valid_index()} for k, v in series_data.items()]).sort_values('series'))

## 2. Apply percentage change to every series

The calculation uses adjacent rows in the cleaned series. A zero previous value makes the ratio undefined and is excluded from the transformed distribution. We also count negative previous values and sign crossings, because percentage changes around zero or negative levels can be mathematically calculable but economically difficult to interpret. No winsorization or clipping is applied.

In [ ]:
percentage_changes = {}
transformation_diagnostics = []

for name, level in series_data.items():
    previous = level.shift(1)
    raw_pct = 100.0 * (level / previous - 1.0)
    zero_denominator = previous.eq(0) & previous.notna() & level.notna()
    negative_denominator = previous.lt(0) & previous.notna() & level.notna()
    sign_crossing = (previous * level).lt(0) & previous.notna() & level.notna()
    # Keep the conventional percentage-change formula only where the prior level is non-zero.
    pct = raw_pct.replace([np.inf, -np.inf], np.nan)
    pct.loc[zero_denominator] = np.nan
    percentage_changes[name] = pct.dropna()
    transformation_diagnostics.append({
        'series': name,
        'level_observations': int(level.notna().sum()),
        'valid_percentage_changes': int(pct.notna().sum()),
        'zero_previous_values': int(zero_denominator.sum()),
        'negative_previous_values': int(negative_denominator.sum()),
        'sign_crossings': int(sign_crossing.sum()),
        'minimum_pct_change': pct.min(),
        'maximum_pct_change': pct.max(),
    })

diagnostics_df = pd.DataFrame(transformation_diagnostics).sort_values('series')
display(diagnostics_df)

## 3. Distribution summary

The table reports location, dispersion, skewness, and excess kurtosis for each percentage-change distribution. These are descriptive diagnostics only. A smoother-looking histogram or lower kurtosis does not by itself establish stationarity, better calibration, or greater economic usefulness.

In [ ]:
summary_rows = []
for name, values in percentage_changes.items():
    x = values.dropna().to_numpy(dtype=float)
    summary_rows.append({
        'series': name,
        'n': len(x),
        'mean': np.mean(x) if len(x) else np.nan,
        'median': np.median(x) if len(x) else np.nan,
        'std_sample': np.std(x, ddof=1) if len(x) > 1 else np.nan,
        'min': np.min(x) if len(x) else np.nan,
        'p01': np.quantile(x, 0.01) if len(x) else np.nan,
        'p05': np.quantile(x, 0.05) if len(x) else np.nan,
        'p25': np.quantile(x, 0.25) if len(x) else np.nan,
        'p75': np.quantile(x, 0.75) if len(x) else np.nan,
        'p95': np.quantile(x, 0.95) if len(x) else np.nan,
        'p99': np.quantile(x, 0.99) if len(x) else np.nan,
        'max': np.max(x) if len(x) else np.nan,
        'skew': stats.skew(x, bias=False) if len(x) > 2 else np.nan,
        'excess_kurtosis': stats.kurtosis(x, fisher=True, bias=False) if len(x) > 3 else np.nan,
        'zero_change_share_pct': 100 * np.mean(np.isclose(x, 0)) if len(x) else np.nan,
    })

summary_df = pd.DataFrame(summary_rows).sort_values('series').reset_index(drop=True)
display(summary_df)

## 4. Plot each series' percentage-change distribution

Each series receives a histogram and boxplot. Histograms use the full valid transformed sample, with no tail trimming. The boxplot helps reveal outliers and asymmetry that may be difficult to see in the histogram.

In [ ]:
for name in sorted(percentage_changes):
    values = percentage_changes[name].dropna()
    if len(values) < 2:
        print(f'Skipping {name}: fewer than two valid percentage changes.')
        continue

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    axes[0].hist(values, bins='auto', edgecolor='black', alpha=0.75)
    axes[0].axvline(values.mean(), linestyle='--', linewidth=1.5, label=f'Mean = {values.mean():.3g}%')
    axes[0].axvline(values.median(), linestyle=':', linewidth=1.5, label=f'Median = {values.median():.3g}%')
    axes[0].set_title(f'{name}: Percentage-Change Distribution')
    axes[0].set_xlabel('Percentage change (%)')
    axes[0].set_ylabel('Frequency')
    axes[0].legend()

    axes[1].boxplot(values, vert=True, showmeans=True)
    axes[1].set_title(f'{name}: Boxplot')
    axes[1].set_ylabel('Percentage change (%)')
    axes[1].set_xticks([1], [name])
    plt.tight_layout()
    plt.show()

## 5. Compare distribution shapes across series

Because series have different scales and units, this comparison standardizes each percentage-change distribution using its own mean and sample standard deviation. It compares shape rather than economic magnitude. Constant or nearly constant series are omitted from this particular plot and remain visible in the summary table.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
plotted = 0
for name in sorted(percentage_changes):
    values = percentage_changes[name].dropna()
    sd = values.std(ddof=1)
    if len(values) < 3 or not np.isfinite(sd) or sd == 0:
        continue
    standardized = (values - values.mean()) / sd
    ax.hist(standardized, bins=60, density=True, histtype='step', linewidth=1.0, label=name)
    plotted += 1
ax.set_title('Standardized Percentage-Change Distributions (shape comparison)')
ax.set_xlabel('Within-series standardized percentage change')
ax.set_ylabel('Density')
if plotted <= 15:
    ax.legend(ncol=2, fontsize=8)
else:
    ax.legend(ncol=4, fontsize=6, bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()
print(f'Series plotted: {plotted}')

## 6. Interpretation checklist

- Compare skewness, excess kurtosis, tail quantiles, and zero-change share against the original `analysis.ipynb` results.
- Inspect the diagnostic counts before interpreting any series with zero or negative previous values.
- Percentage change is not equivalent to log change: `100 * (X_t / X_(t-1) - 1)` is the exact ordinary percentage change, while `100 * ln(X_t / X_(t-1))` is a log change and only approximates percentage change for small movements.
- For rate variables such as unemployment or policy rates, a percentage change can answer a different question from a percentage-point change. This notebook deliberately applies the same formula to all series as an experiment, not as a recommended final transformation registry.
- The first valid change is lost by differencing. No outlier clipping, winsorization, or manual crisis-date exclusion is applied here.

**Research question:** does applying a uniform percentage-change transformation make distributions more comparable, or does it introduce denominator effects and less interpretable movements for some macro series?

## 7. Per-series percentage-change benchmark plots

This section follows the plotting layout in `notebook/benchmarkplotting.ipynb`, but uses the percentage changes already computed above. For each series it plots (1) percentage change centered on its own mean with ±1σ and ±2σ reference bands, and (2) its z-score through time. No earlier cells are modified. These are descriptive statistical references, not economic health thresholds or causal signals.

In [ ]:
from IPython.display import display

# Keep outputs in the same general location as benchmarkplotting.ipynb.
plot_dir = Path('outputs/benchmark/percentage_change_plots') if Path('outputs').exists() else Path('../outputs/benchmark/percentage_change_plots')
plot_dir.mkdir(parents=True, exist_ok=True)

percentage_change_plot_summary = []
for name, values in sorted(percentage_changes.items()):
    change = values.dropna().sort_index()
    if len(change) < 2:
        print(f'Skipping {name}: fewer than two valid percentage changes')
        continue

    center = change.mean()
    scale = change.std(ddof=1)
    if pd.isna(scale) or scale <= 0:
        print(f'Skipping standardized plot for {name}: standard deviation is zero or invalid')
        continue
    centered = change - center
    score = centered / scale

    fig, axes = plt.subplots(2, 1, figsize=(12, 7), constrained_layout=True)
    axes[0].plot(change.index, centered, linewidth=1, label='Centered percentage change')
    for k, alpha in [(1, 0.65), (2, 0.35)]:
        axes[0].axhline(k * scale, linestyle='--', linewidth=1, alpha=alpha, label=f'+{k}σ')
        axes[0].axhline(-k * scale, linestyle='--', linewidth=1, alpha=alpha, label=f'−{k}σ')
    axes[0].axhline(0, color='black', linewidth=1)
    axes[0].set_title(f'{name} — percentage change centered on its mean')
    axes[0].set_ylabel('Percentage points from mean')
    axes[0].legend(ncol=3, fontsize=8)

    axes[1].plot(score.index, score, linewidth=1, label='Percentage-change z-score')
    axes[1].axhline(0, color='black', linewidth=1)
    axes[1].axhline(1, linestyle='--', alpha=0.6, label='+1σ / −1σ')
    axes[1].axhline(-1, linestyle='--', alpha=0.6)
    axes[1].axhline(2, linestyle=':', alpha=0.6, label='+2σ / −2σ')
    axes[1].axhline(-2, linestyle=':', alpha=0.6)
    axes[1].set_title('Standardized percentage change through time')
    axes[1].set_ylabel('Z-score')
    axes[1].set_xlabel('Observation date')
    axes[1].legend(fontsize=8)

    fig.savefig(plot_dir / f'{name}_percentage_change_benchmark.png', dpi=130)
    plt.show()
    plt.close(fig)

    percentage_change_plot_summary.append({
        'series': name,
        'n_changes': int(change.notna().sum()),
        'mean_pct_change': float(center),
        'std_pct_change': float(scale),
        'max_abs_z_score': float(score.abs().max()),
        'date_of_max_abs_z': score.abs().idxmax(),
    })

percentage_change_plot_summary = pd.DataFrame(percentage_change_plot_summary).sort_values('series')
display(percentage_change_plot_summary.round(5))
percentage_change_plot_summary.to_csv(plot_dir / 'percentage_change_plot_statistics.csv', index=False)
print(f'Plots and statistics saved to {plot_dir.resolve()}')